In [ ]:
"""
ARSA Variant Analysis

Machine-learning analysis of ARSA genetic variants using Python and scikit-learn.

The pipeline:
1. Loads and merges the provided training and ARSA datasets.
2. Engineers an MNV indicator and prepares selected prediction features.
3. Cleans and normalizes computational prediction scores.
4. Compares Random Forest and k-nearest neighbors models using 5-fold ROC-AUC.
5. Trains the selected Random Forest model on the full labeled dataset.
6. Generates pathogenicity probabilities for ARSA variants.
7. Converts pathogenicity probabilities into the required stability scores.
8. Writes predictions to the required CAGI submission format.

Note:
The final stability-related prediction is derived from pathogenicity
probabilities rather than being trained directly on experimentally measured
protein stability.
"""

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIGURATION
# ============================================================

RANDOM_STATE = 42
CV_FOLDS = 5

INVERT_FEATURES = [
    "SIFT_score",
    "PROVEAN_score",
    "FATHMM_score",
    "ESM1b_score",
]

STANDARD_FEATURES = [
    "Polyphen2_HVAR_score",
    "CADD_raw",
    "GERP++_RS",
    "AlphaMissense_score",
    "is_MNV",
]

ALL_FEATURES = INVERT_FEATURES + STANDARD_FEATURES

LABEL_MAPPING = {
    "Pathogenic": 1,
    "Benign": 0,
}


# ============================================================
# DATA LOADING
# ============================================================

def load_data(
    training_path,
    arsa_sample_path,
    target_path,
    template_path,
):
    """
    Load the provided datasets and merge the main training data
    with the ARSA-specific sample.

    Parameters
    ----------
    training_path : str or Path
        Path to the main labeled training dataset.

    arsa_sample_path : str or Path
        Path to the ARSA-specific sample dataset.

    target_path : str or Path
        Path to the ARSA variants requiring predictions.

    template_path : str or Path
        Path to the final CAGI submission template.

    Returns
    -------
    df_train : pandas.DataFrame
        Combined training dataset.

    df_target : pandas.DataFrame
        Target ARSA variants.

    template : pandas.DataFrame
        Submission template.
    """

    print("Loading data...")

    df_train_main = pd.read_csv(training_path, sep="\t")
    df_arsa_sample = pd.read_csv(arsa_sample_path, sep="\t")
    df_target = pd.read_csv(target_path, sep="\t")
    template = pd.read_csv(template_path, sep="\t")

    # Remove hidden whitespace from column names.
    for df in [df_train_main, df_arsa_sample, df_target]:
        df.columns = df.columns.str.strip()

    # Combine the main training data with the ARSA-specific sample.
    df_train = pd.concat(
        [df_train_main, df_arsa_sample],
        axis=0,
        ignore_index=True,
    )

    return df_train, df_target, template


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def add_mnv_flag(df):
    """
    Add an indicator identifying multi-nucleotide variants (MNVs).

    A variant is flagged as an MNV if either the reference or
    alternate sequence contains more than one character.

    Parameters
    ----------
    df : pandas.DataFrame
        Variant dataset containing 'ref' and 'alt' columns.

    Returns
    -------
    pandas.DataFrame
        Copy of the input DataFrame with an added 'is_MNV' column.
    """

    df = df.copy()

    df["is_MNV"] = df.apply(
        lambda row: (
            1
            if len(str(row["ref"])) > 1
            or len(str(row["alt"])) > 1
            else 0
        ),
        axis=1,
    )

    return df


def select_features(df):
    """
    Select the features used by the machine-learning models.

    Parameters
    ----------
    df : pandas.DataFrame
        Variant dataset containing the engineered features.

    Returns
    -------
    pandas.DataFrame
        Copy containing only the selected model features.
    """

    return df[ALL_FEATURES].copy()


# ============================================================
# FEATURE PREPROCESSING
# ============================================================

def force_numeric(df_features):
    """
    Convert prediction features to numeric values.

    When multiple transcript-specific values are provided in a
    single field, only the first value is retained. Remaining
    non-numeric values are converted to NaN for downstream
    median imputation.

    Parameters
    ----------
    df_features : pandas.DataFrame
        Selected model features.

    Returns
    -------
    pandas.DataFrame
        Numerically cleaned feature DataFrame.
    """

    df_features = df_features.copy()

    for column in ALL_FEATURES:
        if column == "is_MNV":
            continue

        # Keep only the first transcript-specific value.
        df_features[column] = (
            df_features[column]
            .astype(str)
            .str.split(";")
            .str[0]
            .str.split(",")
            .str[0]
        )

        # Convert invalid or empty values to NaN.
        df_features[column] = pd.to_numeric(
            df_features[column],
            errors="coerce",
        )

    return df_features


def invert_scores(df_features):
    """
    Invert prediction scores for which lower values indicate
    greater pathogenicity.

    This places the selected features on a consistent direction,
    where higher values correspond to greater pathogenicity.

    Parameters
    ----------
    df_features : pandas.DataFrame
        Numerically cleaned model features.

    Returns
    -------
    pandas.DataFrame
        Feature DataFrame with selected scores inverted.
    """

    df_features = df_features.copy()

    for column in INVERT_FEATURES:
        df_features[column] = df_features[column] * -1

    return df_features


def prepare_features(df):
    """
    Engineer, select, clean, and normalize model features.

    Parameters
    ----------
    df : pandas.DataFrame
        Variant dataset.

    Returns
    -------
    pandas.DataFrame
        Prepared model features.
    """

    df = add_mnv_flag(df)
    features = select_features(df)
    features = force_numeric(features)
    features = invert_scores(features)

    return features


def prepare_labels(df, features):
    """
    Convert clinical significance labels into binary targets and
    remove variants without a recognized label.

    Pathogenic = 1
    Benign = 0

    Parameters
    ----------
    df : pandas.DataFrame
        Training dataset containing 'Clinical_significance'.

    features : pandas.DataFrame
        Prepared training features.

    Returns
    -------
    X : pandas.DataFrame
        Features corresponding to variants with valid labels.

    y : pandas.Series
        Binary pathogenicity labels.
    """

    y = df["Clinical_significance"].map(LABEL_MAPPING)

    valid_indices = y.dropna().index

    X = features.loc[valid_indices].copy()
    y = y.loc[valid_indices].astype(int)

    return X, y


# ============================================================
# MODELING
# ============================================================

def build_models():
    """
    Construct the models used for comparison.

    Returns
    -------
    dict
        Dictionary containing the Random Forest and k-NN models.
    """

    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=RANDOM_STATE,
        ),
        "k-NN": KNeighborsClassifier(
            n_neighbors=15,
            weights="distance",
        ),
    }


def build_pipeline(model):
    """
    Build the preprocessing and modeling pipeline.

    Missing values are median-imputed and features are standardized
    before being passed to the classifier.

    Parameters
    ----------
    model : scikit-learn estimator
        Classification model.

    Returns
    -------
    sklearn.pipeline.Pipeline
        Complete preprocessing and modeling pipeline.
    """

    return Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                model,
            ),
        ]
    )


def compare_models(X, y, cv_folds=CV_FOLDS):
    """
    Compare candidate classification models using cross-validation.

    ROC-AUC is used as the evaluation metric.

    Parameters
    ----------
    X : pandas.DataFrame
        Training features.

    y : pandas.Series
        Binary pathogenicity labels.

    cv_folds : int, default=5
        Number of cross-validation folds.

    Returns
    -------
    pandas.DataFrame
        Model comparison results.
    """

    print(
        f"Data ready. Total training variants: {X.shape[0]}. "
        f"Running {cv_folds}-fold CV comparison..."
    )

    models = build_models()
    comparison_results = []

    for model_name, model in models.items():
        pipeline = build_pipeline(model)

        cv_scores = cross_validate(
            pipeline,
            X,
            y,
            cv=cv_folds,
            scoring="roc_auc",
            error_score="raise",
            n_jobs=-1,
        )

        comparison_results.append(
            {
                "Model": model_name,
                "Mean CV ROC-AUC": cv_scores["test_score"].mean(),
            }
        )

    results = pd.DataFrame(comparison_results)

    print("\n--- Model Comparison ---")
    print(results.to_string(index=False))

    return results


# ============================================================
# FINAL PREDICTION
# ============================================================

def train_final_model(X, y):
    """
    Train the final Random Forest model on the full labeled dataset.

    Parameters
    ----------
    X : pandas.DataFrame
        Training features.

    y : pandas.Series
        Binary pathogenicity labels.

    Returns
    -------
    sklearn.pipeline.Pipeline
        Fitted Random Forest pipeline.
    """

    print("\nTraining final Random Forest model...")

    models = build_models()
    final_model = build_pipeline(models["Random Forest"])

    final_model.fit(X, y)

    return final_model


def generate_predictions(model, X_target):
    """
    Generate stability-related predictions from the fitted model.

    The Random Forest first estimates pathogenicity probability.
    The required stability score is calculated as:

        Stability = 1 - Pathogenicity

    Parameters
    ----------
    model : sklearn.pipeline.Pipeline
        Fitted Random Forest pipeline.

    X_target : pandas.DataFrame
        Prepared target features.

    Returns
    -------
    numpy.ndarray
        Predicted stability scores.
    """

    pathogenicity_probabilities = model.predict_proba(X_target)[:, 1]

    stability_scores = 1 - pathogenicity_probabilities

    return stability_scores


# ============================================================
# SUBMISSION
# ============================================================

def create_submission(template, stability_scores, output_path):
    """
    Populate the CAGI submission template and save it as a TSV.

    Parameters
    ----------
    template : pandas.DataFrame
        Original submission template.

    stability_scores : numpy.ndarray
        Predicted stability scores.

    output_path : str or Path
        Destination path for the submission file.

    Returns
    -------
    pandas.DataFrame
        Completed submission DataFrame.
    """

    submission = template.copy()

    submission["stability_score_48hr"] = stability_scores
    submission["sd"] = "*"
    submission["comment"] = ""

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    submission.to_csv(
        output_path,
        sep="\t",
        index=False,
    )

    print(
        f"\nFinal predictions complete! "
        f"Saved submission to: '{output_path}'"
    )

    return submission


# ============================================================
# MAIN ANALYSIS PIPELINE
# ============================================================

def main():
    """
    Run the complete ARSA variant analysis pipeline.
    """

    # --------------------------------------------------------
    # File paths
    # --------------------------------------------------------

    project_root = Path(__file__).resolve().parents[1]

    data_dir = project_root / "Data"
    results_dir = project_root / "Results"

    training_path = data_dir / "s26-c146-project-training.tsv"
    arsa_sample_path = (
        data_dir / "s26-c146-project-cagi-7-arsa-sample.tsv"
    )
    target_path = (
        data_dir / "s26-c146-arsa-cagi-snv-features.tsv"
    )
    template_path = (
        data_dir / "s26-c146-arsa-cagi-snv-template.tsv"
    )

    submission_path = (
        results_dir / "s26-c146-arsa-cagi-snv-submission.tsv"
    )

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    df_train, df_target, template = load_data(
        training_path=training_path,
        arsa_sample_path=arsa_sample_path,
        target_path=target_path,
        template_path=template_path,
    )

    # --------------------------------------------------------
    # Prepare features
    # --------------------------------------------------------

    print("Preparing model features...")

    X_train_raw = prepare_features(df_train)
    X_target = prepare_features(df_target)

    X_train, y_train = prepare_labels(
        df_train,
        X_train_raw,
    )

    print(
        f"Training data prepared: "
        f"{X_train.shape[0]} labeled variants, "
        f"{X_train.shape[1]} features."
    )

    # --------------------------------------------------------
    # Compare models
    # --------------------------------------------------------

    comparison_results = compare_models(
        X_train,
        y_train,
        cv_folds=CV_FOLDS,
    )

    # --------------------------------------------------------
    # Train final model
    # --------------------------------------------------------

    final_model = train_final_model(
        X_train,
        y_train,
    )

    # --------------------------------------------------------
    # Generate ARSA predictions
    # --------------------------------------------------------

    stability_scores = generate_predictions(
        final_model,
        X_target,
    )

    # --------------------------------------------------------
    # Create submission
    # --------------------------------------------------------

    submission = create_submission(
        template=template,
        stability_scores=stability_scores,
        output_path=submission_path,
    )

    # --------------------------------------------------------
    # Basic validation
    # --------------------------------------------------------

    print("\n--- Submission Validation ---")

    expected_rows = len(template)
    actual_rows = len(submission)

    missing_stability = submission[
        "stability_score_48hr"
    ].isna().sum()

    print(f"Expected rows: {expected_rows}")
    print(f"Submitted rows: {actual_rows}")
    print(f"Missing stability scores: {missing_stability}")

    if expected_rows != actual_rows:
        raise ValueError(
            "Submission row count does not match the template."
        )

    if missing_stability > 0:
        raise ValueError(
            "Submission contains missing stability scores."
        )

    print("Validation complete.")


if __name__ == "__main__":
    main()

Loading data...
Flagging MNVs and defining features...
Cleaning data and handling multiple transcript scores...
Data ready. Total training variants: 13464. Running 5-fold CV Comparison...

--- Final Model Comparison ---
           Model  Mean CV ROC-AUC
0  Random Forest         0.940374
1           k-NN         0.912302

Training final Meta-Predictor on full dataset...
Final predictions complete! Saved filled template to: 's26-c146-arsa-cagi-snv-submission.tsv'


In [ ]:
!python s26-c146-arsa-cagi-snv-validation.py s26-c146-arsa-cagi-snv-submission.tsv s26-c146-arsa-cagi-snv-template.tsv


Expected variants: 2491
Submitted variants: 2491
Missing variants: 0
Warnings: 0
Errors: 0

The file's format is valid and complete! You are good to submit now!
